# IE643 — EDA: CheXpert Plus 10k stratified subset

Dataset: **IE643 Project, Version 2** (`jithendraampatt/ie643-project`) — 10,000 frontal chest X-rays
(3,325 *No Finding* + 6,675 abnormal, `random_state=42`) with their radiology reports.

What this notebook answers, in order:
1. **Integrity** — is every image the manifest promises actually here?
2. **Who is in it** — class balance, patients with several studies (split leakage), demographics.
3. **What we generate** — findings vs impression availability, report length in tokens, how templated the text is.
4. **What we feed the model** — image size, bit depth, intensity, file size.
5. **A leakage-free train/val/test split**, saved to `/kaggle/working/splits.csv`.

Attach *IE643 Project → Version 2* as input. No GPU needed. Internet on is optional (only used for the GPT-2 tokenizer).

In [ ]:
import os, re, glob, json, textwrap, collections, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

pd.set_option("display.max_colwidth", 120)
NORMAL, ABNORMAL = "#2a78d6", "#eb6834"          # validated colour pair (CVD-safe)
CLS_COL = {"Normal": NORMAL, "Abnormal": ABNORMAL}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "axes.titleweight": "bold"})
SEED = 42
rng = np.random.default_rng(SEED)

def find_root(base="/kaggle/input", max_depth=4):
    # Directory holding stratified_manifest.csv (works for old and new Kaggle mount layouts)
    for dirpath, dirnames, filenames in os.walk(base):
        if "stratified_manifest.csv" in filenames:
            return dirpath
        depth = dirpath[len(base):].count(os.sep)
        dirnames[:] = [] if depth >= max_depth else [d for d in dirnames if not d.startswith(("part_", "patient"))]
    raise FileNotFoundError("Attach 'IE643 Project' Version 2 as an input")

ROOT = find_root()
df = pd.read_csv(os.path.join(ROOT, "stratified_manifest.csv"))
df["label"] = np.where(df["is_no_finding"].astype(bool), "Normal", "Abnormal")
df["abs_path"] = df["local_path"].map(lambda p: os.path.join(ROOT, p))
print("ROOT:", ROOT)
print("Manifest:", df.shape)

## 1. Schema and missingness

In [ ]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing_%": (df.isna().mean() * 100).round(1),
    "unique": df.nunique(),
    "example": df.iloc[0].astype(str).str.slice(0, 60),
})
schema

In [ ]:
for c in ["split", "frontal_lateral", "sex", "race", "ethnicity", "insurance_type"]:
    if c in df:
        print(f"\n{c}:\n" + df[c].fillna("<missing>").value_counts().head(8).to_string())

## 2. Integrity — every image present, readable path, consistent size

In [ ]:
exists = df["abs_path"].map(os.path.exists)
on_disk = {os.path.join(r, f) for r, _, fs in os.walk(ROOT) for f in fs if f.lower().endswith(".png")}
disk_bytes = df.loc[exists, "abs_path"].map(os.path.getsize)

print(f"Rows in manifest           : {len(df):,}")
print(f"Manifest paths present     : {exists.sum():,}   missing: {(~exists).sum():,}")
print(f"PNG files on disk          : {len(on_disk):,}   not in manifest: {len(on_disk - set(df['abs_path'])):,}")
print(f"Duplicate local paths      : {df['local_path'].duplicated().sum():,}")
print(f"Duplicate source images    : {df['path_to_image'].duplicated().sum():,}")
if "size" in df:
    mism = (disk_bytes != df.loc[exists, "size"]).sum()
    print(f"Size ≠ Redivis 'size' col  : {mism:,}")
print(f"Total on disk              : {disk_bytes.sum()/1e9:.2f} GB")
df["part"] = df["local_path"].str.split("/").str[0]
print("\nImages per part:", df["part"].value_counts().sort_index().agg(["min", "max"]).to_dict())

## 3. Class balance and patients

In [ ]:
cls = df["label"].value_counts().reindex(["Normal", "Abnormal"])
per_pat = df.groupby("deid_patient_id").agg(n_img=("local_path", "size"),
                                             n_cls=("label", "nunique"))
print(cls.to_string(), f"\nAbnormal : Normal = {cls['Abnormal']/cls['Normal']:.2f} : 1\n")
print(f"Patients                    : {len(per_pat):,}")
print(f"Patients with >1 image      : {(per_pat.n_img > 1).sum():,} "
      f"({(per_pat.n_img > 1).mean()*100:.1f}%), covering {per_pat.loc[per_pat.n_img>1,'n_img'].sum():,} images")
print(f"Patients in BOTH classes    : {(per_pat.n_cls > 1).sum():,}")
print(f"Max images for one patient  : {per_pat.n_img.max()}")
print("→ An image-level random split would put the same patient in train and test. Split by patient (section 6).")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].bar(cls.index, cls.values, color=[CLS_COL[k] for k in cls.index], width=.55)
for i, v in enumerate(cls.values):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")
ax[0].set_title("Images per class"); ax[0].set_ylabel("images")
vc = per_pat.n_img.value_counts().sort_index()
ax[1].bar(vc.index.astype(str), vc.values, color=NORMAL, width=.6)
ax[1].set_yscale("log"); ax[1].set_title("Images per patient"); ax[1].set_xlabel("images"); ax[1].set_ylabel("patients (log)")
plt.tight_layout(); plt.show()

## 4. Demographics by class

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
bins = np.arange(15, 95, 5)
for lab in ["Normal", "Abnormal"]:
    a = df.loc[df.label == lab, "age"].dropna()
    ax[0].hist(a, bins=bins, histtype="step", lw=2, color=CLS_COL[lab], density=True, label=f"{lab} (median {a.median():.0f})")
ax[0].set_title("Age"); ax[0].set_xlabel("years"); ax[0].legend(frameon=False)

sx = pd.crosstab(df["sex"].fillna("Unknown"), df["label"], normalize="columns")[["Normal", "Abnormal"]] * 100
sx.plot.bar(ax=ax[1], color=[NORMAL, ABNORMAL], width=.7, rot=0); ax[1].set_title("Sex (% within class)"); ax[1].set_xlabel("")
ax[1].legend(frameon=False)

if "recent_bmi" in df and df["recent_bmi"].notna().any():
    for lab in ["Normal", "Abnormal"]:
        b = df.loc[df.label == lab, "recent_bmi"].dropna().clip(10, 60)
        ax[2].hist(b, bins=30, histtype="step", lw=2, color=CLS_COL[lab], density=True, label=lab)
    ax[2].set_title(f"Recent BMI ({df['recent_bmi'].notna().mean()*100:.0f}% available)"); ax[2].legend(frameon=False)
else:
    ax[2].axis("off")
plt.tight_layout(); plt.show()

print(df.groupby("label")["age"].describe().round(1))

## 5. Report text — the generation target

CheXpert Plus reports are split into sections. The project's teacher prompt targets the **impression**;
findings are only kept if they turn out to be present often enough to matter.

In [ ]:
def clean(s):
    return re.sub(r"\s+", " ", str(s)).strip() if isinstance(s, str) else np.nan

for c in ["section_findings", "section_impression"]:
    df[c + "_clean"] = df[c].map(clean)
has_f = df["section_findings_clean"].notna() & (df["section_findings_clean"].str.len() > 0)
has_i = df["section_impression_clean"].notna() & (df["section_impression_clean"].str.len() > 0)
print(pd.crosstab(has_f.rename("has findings"), has_i.rename("has impression"), margins=True))
print(f"\nImpression available: {has_i.mean()*100:.1f}%   Findings available: {has_f.mean()*100:.1f}%")
print(pd.crosstab(df["label"], has_f.rename("has findings"), normalize="index").round(3) * 100)

In [ ]:
imp = df.loc[has_i, "section_impression_clean"]
words = imp.str.split().str.len()

try:                                     # token counts in the decoder's own tokenizer
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("distilgpt2")
    toks = imp.map(lambda s: len(tok(s)["input_ids"]))
    tok_name = "DistilGPT2 tokens"
except Exception as e:
    toks, tok_name = (words * 1.35).round(), "≈ tokens (words × 1.35; tokenizer unavailable)"
    print("Tokenizer not loaded:", type(e).__name__)

q = [.5, .9, .95, .99, 1.0]
print("Impression words :", words.quantile(q).round(0).astype(int).to_dict())
print(f"Impression {tok_name}:", toks.quantile(q).round(0).astype(int).to_dict())
for L in (64, 96, 128, 160):
    print(f"  max_length={L:<4} truncates {(toks > L).mean()*100:5.1f}% of impressions")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
for lab in ["Normal", "Abnormal"]:
    m = df.loc[has_i, "label"] == lab
    ax[0].hist(toks[m].clip(upper=200), bins=50, histtype="step", lw=2, color=CLS_COL[lab],
               label=f"{lab} (median {toks[m].median():.0f})")
ax[0].set_title("Impression length"); ax[0].set_xlabel(tok_name + " (clipped at 200)"); ax[0].legend(frameon=False)
f_words = df.loc[has_f, "section_findings_clean"].str.split().str.len()
ax[1].hist(f_words.clip(upper=300), bins=50, color=NORMAL)
ax[1].set_title(f"Findings length (n={has_f.sum():,})"); ax[1].set_xlabel("words (clipped at 300)")
plt.tight_layout(); plt.show()

In [ ]:
# How templated is the text? (drives BLEU inflation and mode collapse risk in the student)
norm = imp.str.upper().str.replace(r"^\s*1\s*\.\s*", "", regex=True).str.replace(r"[^A-Z ]", " ", regex=True) \
          .str.replace(r"\s+", " ", regex=True).str.strip()
top = norm.value_counts()
print(f"Unique impressions (normalised): {top.size:,} of {len(norm):,}  "
      f"→ top-10 cover {top.head(10).sum()/len(norm)*100:.1f}% of reports")
print(top.head(15).to_string())

print("\nFormatting:")
print(f"  numbered list ('1.' …)  : {imp.str.match(r'^\s*1\s*\.').mean()*100:.1f}%")
print(f"  mostly upper-case        : {(imp.str.count(r'[A-Z]') > 3*imp.str.count(r'[a-z]')).mean()*100:.1f}%")
for pat in ["COMPARISON", "DICTATED", "SIGNED", "ATTENDING", "I HAVE PERSONALLY", "PRELIMINARY"]:
    r = imp.str.upper().str.contains(pat).mean()*100
    if r > 0.5: print(f"  contains '{pat}': {r:.1f}%")

def ngrams(series, n):
    c = collections.Counter()
    for s in series:
        w = s.split(); c.update(" ".join(w[i:i+n]) for i in range(len(w)-n+1))
    return c
for lab in ["Normal", "Abnormal"]:
    s = norm[df.loc[has_i, "label"] == lab]
    print(f"\nTop trigrams — {lab}:", ", ".join(f"{g} ({k})" for g, k in ngrams(s, 3).most_common(8)))
vocab = collections.Counter(w for s in norm for w in s.split())
print(f"\nVocabulary (upper-cased words): {len(vocab):,};  words seen ≥5 times: {sum(v>=5 for v in vocab.values()):,}")

## 6. Images — what the vision encoder will receive

In [ ]:
N_SAMPLE = 400                                   # full decode of 10k PNGs takes ~10 min; a stratified sample is enough
samp = df.groupby("label").sample(n=N_SAMPLE // 2, random_state=SEED)
rows = []
for p, lab in zip(samp["abs_path"], samp["label"]):
    with Image.open(p) as im:
        a = np.asarray(im)
        rows.append(dict(label=lab, mode=im.mode, w=im.width, h=im.height, dtype=str(a.dtype),
                         bits=a.dtype.itemsize*8, min=int(a.min()), max=int(a.max()),
                         mean=float(a.mean()), std=float(a.std()), mb=os.path.getsize(p)/1e6))
img = pd.DataFrame(rows)
img["aspect"] = img.w / img.h
print("Mode / dtype:", img.groupby(["mode", "dtype"]).size().to_dict())
print(img[["w", "h", "aspect", "min", "max", "mean", "std", "mb"]].describe().round(2))

fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
ax[0].scatter(img.w, img.h, s=10, c=img.label.map(CLS_COL), alpha=.6); ax[0].set_title("Width × height (px)")
ax[0].set_xlabel("width"); ax[0].set_ylabel("height")
scale = 255 if img.bits.max() <= 8 else 65535
for lab in ["Normal", "Abnormal"]:
    m = img.label == lab
    ax[1].hist(img.loc[m, "mean"]/scale, bins=30, histtype="step", lw=2, color=CLS_COL[lab], label=lab)
    ax[2].hist(img.loc[m, "std"]/scale, bins=30, histtype="step", lw=2, color=CLS_COL[lab], label=lab)
ax[1].set_title("Mean intensity (0–1)"); ax[2].set_title("Contrast: std of intensity (0–1)")
for a in ax[1:]: a.legend(frameon=False)
plt.tight_layout(); plt.show()

In [ ]:
show = pd.concat([df[df.label == l].sample(4, random_state=SEED) for l in ["Normal", "Abnormal"]])
fig, axes = plt.subplots(2, 4, figsize=(15, 9))
for a, (_, r) in zip(axes.ravel(), show.iterrows()):
    with Image.open(r.abs_path) as im:
        im.thumbnail((512, 512)); a.imshow(np.asarray(im), cmap="gray")
    a.set_title(f"{r.label} · {r.sex}, {r.age:.0f}y", color=CLS_COL[r.label], fontsize=10)
    a.set_xlabel(textwrap.fill(str(r.section_impression_clean)[:220], 48), fontsize=7.5)
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.tight_layout(); plt.show()

## 7. Leakage-free split by patient

Every patient goes to exactly one of train / val / test (80 / 10 / 10 by image count), stratified so each
split keeps the ~1 : 2 normal : abnormal ratio. Rows without an impression are kept but flagged
(`has_impression`) so the training loader can drop them.

In [ ]:
pat = df.groupby("deid_patient_id").agg(n=("local_path", "size"), any_abn=("label", lambda s: (s == "Abnormal").any()))
assign = {}
for flag, g in pat.groupby("any_abn"):
    ids = g.sample(frac=1, random_state=SEED).index.to_numpy()
    cum = g.loc[ids, "n"].cumsum() / g["n"].sum()
    assign.update({i: ("train" if c <= .8 else "val" if c <= .9 else "test") for i, c in zip(ids, cum)})
df["subset"] = df["deid_patient_id"].map(assign)
df["has_impression"] = has_i

overlap = df.groupby("deid_patient_id")["subset"].nunique().gt(1).sum()
summary = pd.crosstab(df["subset"], df["label"], margins=True)
summary["abnormal_%"] = (summary["Abnormal"] / summary["All"] * 100).round(1)
print(summary.loc[["train", "val", "test", "All"]])
print(f"\nPatients in more than one subset: {overlap}")
assert overlap == 0

out = df[["local_path", "path_to_image", "deid_patient_id", "label", "subset", "has_impression",
          "section_impression_clean", "section_findings_clean", "age", "sex"]]
out.to_csv("/kaggle/working/splits.csv", index=False)
print("Saved /kaggle/working/splits.csv", out.shape)

## 8. Key numbers for the report

In [ ]:
facts = {
    "images": len(df), "patients": df.deid_patient_id.nunique(),
    "normal / abnormal": f"{cls['Normal']:,} / {cls['Abnormal']:,}",
    "patients with >1 image": int((per_pat.n_img > 1).sum()),
    "impression available %": round(has_i.mean()*100, 1),
    "findings available %": round(has_f.mean()*100, 1),
    f"impression {tok_name} p50 / p95 / p99": "/".join(str(int(x)) for x in toks.quantile([.5, .95, .99])),
    "top-10 impressions cover %": round(top.head(10).sum()/len(norm)*100, 1),
    "image size (median w×h)": f"{int(img.w.median())}×{int(img.h.median())}",
    "pixel depth": ", ".join(sorted(set(img["mode"] + "/" + img["dtype"]))),
    "dataset size GB": round(disk_bytes.sum()/1e9, 2),
    "split images train/val/test": "/".join(str(int((df.subset == s).sum())) for s in ["train", "val", "test"]),
}
pd.Series(facts, name="value").to_frame()